In [2]:
!pip install --upgrade mediapipe==0.10.14 protobuf==4.25.3 -q

import os
import cv2
import numpy as np
import mediapipe as mp
import shutil
from glob import glob
from tqdm import tqdm
from pathlib import Path

BODY_LANDMARKS = [
    "nose", "leftEyeInner", "leftEye", "leftEyeOuter", "rightEyeInner", "rightEye", "rightEyeOuter",
    "leftEar", "rightEar", "mouthLeft", "mouthRight", "leftShoulder", "rightShoulder",
    "leftElbow", "rightElbow", "leftWrist", "rightWrist", "leftPinky", "rightPinky",
    "leftIndex", "rightIndex", "leftThumb", "rightThumb", 
    "leftHip", "rightHip", "leftKnee", "rightKnee", "leftAnkle", "rightAnkle", 
    "leftHeel", "rightHeel", "leftFootIndex", "rightFootIndex", "neck"
]

HAND_LANDMARKS = [
    "wrist", "indexTip", "indexDIP", "indexPIP", "indexMCP",
    "middleTip", "middleDIP", "middlePIP", "middleMCP",
    "ringTip", "ringDIP", "ringPIP", "ringMCP",
    "littleTip", "littleDIP", "littlePIP", "littleMCP",
    "thumbTip", "thumbIP", "thumbMP", "thumbCMC",
]

HANDS_LANDMARKS = [id + suffix for id in HAND_LANDMARKS for suffix in ["_0", "_1"]]
LANDMARKS = BODY_LANDMARKS + HANDS_LANDMARKS

print(f"✅ Đã cấu hình danh mục! Tổng: {len(LANDMARKS)} điểm khớp (34 Body + 42 Hands).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 15.8 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 12.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
a2a-sdk 0.3.25 requires protobuf>=5.29.5, but you have protobuf 4.25.3 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 4.25.3 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.3 which is incompatible.
google-api-core 2.30.0 requires protobuf<7.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf

2026-05-17 15:23:39.873157: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779031420.083594      56 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779031420.152548      56 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779031420.677040      56 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779031420.677089      56 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779031420.677092      56 computation_placer.cc:177] computation placer alr

✅ Đã cấu hình danh mục! Tổng: 76 điểm khớp (34 Body + 42 Hands).


In [3]:
import os
import cv2
import numpy as np
import mediapipe as mp

mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(static_image_mode=False, model_complexity=1)
mp_drawing  = mp.solutions.drawing_utils

class SingleBodyDictNormalize:
    def __call__(self, row: dict) -> dict:
        sequence_size = len(row["leftEar"])
        ANCHOR_LANDMARKS = ["nose", "leftShoulder", "rightShoulder", "leftHip", "rightHip", "neck"]
        
        for i in range(sequence_size):
            x_coords = [row[name][i][0] for name in ANCHOR_LANDMARKS if row[name][i][0] != 0]
            y_coords = [row[name][i][1] for name in ANCHOR_LANDMARKS if row[name][i][1] != 0]
            
            if not x_coords or not y_coords: continue
            
            min_x, max_x = min(x_coords), max(x_coords)
            min_y, max_y = min(y_coords), max(y_coords)
            
            dx = (max_x - min_x) * 1.6
            dy = (max_y - min_y) * 1.6
            if dx <= 0 or dy <= 0: continue
            
            center_x = (max_x + min_x) / 2
            center_y = (max_y + min_y) / 2
            
            box_min_x = center_x - dx / 2
            box_min_y = center_y - dy / 2
            
            for key in BODY_LANDMARKS:
                x, y, z = row[key][i]
                if x == 0 and y == 0: continue  
                row[key][i] = ((x - box_min_x) / dx - 0.5, (y - box_min_y) / dy - 0.5, z)
        return row

class SingleHandDictNormalize:
    def __call__(self, row: dict) -> dict:
        sequence_size = len(row["leftEar"])
        for suffix in ["_0", "_1"]:
            for i in range(sequence_size):
                x_coords = [row[name + suffix][i][0] for name in HAND_LANDMARKS if row[name + suffix][i][0] != 0]
                y_coords = [row[name + suffix][i][1] for name in HAND_LANDMARKS if row[name + suffix][i][1] != 0]
                if not x_coords or not y_coords: continue
                min_x, max_x = min(x_coords), max(x_coords)
                min_y, max_y = min(y_coords), max(y_coords)
                dx, dy = max_x - min_x, max_y - min_y
                if dx <= 0 or dy <= 0: continue
                for name in HAND_LANDMARKS:
                    x, y, z = row[name + suffix][i]
                    if x == 0 and y == 0: continue
                    row[name + suffix][i] = ((x - min_x) / dx - 0.5, (y - min_y) / dy - 0.5, z)
        return row

body_norm = SingleBodyDictNormalize()
hand_norm = SingleHandDictNormalize()

def extract_sign_language_features(video_path, npy_out_path, visualize=False, vis_out_path=None):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return False
    fps = cap.get(cv2.CAP_PROP_FPS)
    orig_w, orig_h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    video_writer = None
    if visualize and vis_out_path:
        pane_w = 400
        pane_h = int(pane_w * (orig_h / orig_w))
        os.makedirs(os.path.dirname(vis_out_path), exist_ok=True)
        video_writer = cv2.VideoWriter(vis_out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (pane_w * 3, pane_h))
        
    video_sequence_dict = {name: [] for name in LANDMARKS}
    
    while True:
        ret, frame = cap.read()
        if not ret: break
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb_frame)
        
        if visualize and video_writer:
            pane_original = cv2.resize(frame, (pane_w, pane_h))
            frame_skeleton_only = np.zeros_like(frame)
            frame_overlay = frame.copy()
            def draw_mp_skeleton(img):
                if results.pose_landmarks: mp_drawing.draw_landmarks(img, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
                if results.left_hand_landmarks: mp_drawing.draw_landmarks(img, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
                if results.right_hand_landmarks: mp_drawing.draw_landmarks(img, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
            draw_mp_skeleton(frame_skeleton_only); draw_mp_skeleton(frame_overlay)
            pane_skeleton = cv2.resize(frame_skeleton_only, (pane_w, pane_h))
            pane_overlay = cv2.resize(frame_overlay, (pane_w, pane_h))
            combined_pane = np.hstack((pane_original, pane_skeleton, pane_overlay))
            video_writer.write(combined_pane)
            
        pose_data = {name: (0.0, 0.0, 0.0) for name in BODY_LANDMARKS}
        if results.pose_landmarks:
            pose_map = {
                "nose": 0, "leftEyeInner": 1, "leftEye": 2, "leftEyeOuter": 3, "rightEyeInner": 4, "rightEye": 5, "rightEyeOuter": 6,
                "leftEar": 7, "rightEar": 8, "mouthLeft": 9, "mouthRight": 10, "leftShoulder": 11, "rightShoulder": 12,
                "leftElbow": 13, "rightElbow": 14, "leftWrist": 15, "rightWrist": 16, "leftPinky": 17, "rightPinky": 18,
                "leftIndex": 19, "rightIndex": 20, "leftThumb": 21, "rightThumb": 22, "leftHip": 23, "rightHip": 24,
                "leftKnee": 25, "rightKnee": 26, "leftAnkle": 27, "rightAnkle": 28, "leftHeel": 29, "rightHeel": 30,
                "leftFootIndex": 31, "rightFootIndex": 32
            }
            for name, idx in pose_map.items():
                lm = results.pose_landmarks.landmark[idx]
                pose_data[name] = (lm.x, lm.y, lm.z)
                
            ls, rs = pose_data["leftShoulder"], pose_data["rightShoulder"]
            pose_data["neck"] = ((ls[0] + rs[0])/2, (ls[1] + rs[1])/2, (ls[2] + rs[2])/2)
            
        for name in BODY_LANDMARKS: video_sequence_dict[name].append(pose_data[name])
            
        for suffix, hand_landmarks in [("_0", results.left_hand_landmarks), ("_1", results.right_hand_landmarks)]:
            hand_data = {name + suffix: (0.0, 0.0, 0.0) for name in HAND_LANDMARKS}
            if hand_landmarks:
                hand_map = {"wrist": 0, "thumbCMC": 1, "thumbMP": 2, "thumbIP": 3, "thumbTip": 4, "indexMCP": 5, "indexPIP": 6, "indexDIP": 7, "indexTip": 8, "middleMCP": 9, "middlePIP": 10, "middleDIP": 11, "middleTip": 12, "ringMCP": 13, "ringPIP": 14, "ringDIP": 15, "ringTip": 16, "littleMCP": 17, "littlePIP": 18, "littleDIP": 19, "littleTip": 20}
                for name, idx in hand_map.items():
                    lm = hand_landmarks.landmark[idx]
                    hand_data[name + suffix] = (lm.x, lm.y, lm.z)
            for name in HAND_LANDMARKS: video_sequence_dict[name + suffix].append(hand_data[name + suffix])
                
    cap.release()
    if video_writer: video_writer.release()
        
    video_sequence_dict = body_norm(video_sequence_dict)
    video_sequence_dict = hand_norm(video_sequence_dict)
    
    frames_list = []
    seq_len = len(video_sequence_dict["neck"])
    if seq_len == 0: return False
    for i in range(seq_len):
        frame_features = [video_sequence_dict[name][i] for name in LANDMARKS]
        frames_list.append(frame_features)
        
    np_array_data = np.array(frames_list, dtype=np.float32)
    os.makedirs(os.path.dirname(npy_out_path), exist_ok=True)
    np.save(npy_out_path, np_array_data)
    return True

print("✅ Pipeline core trích xuất 3D hoạt động ổn định!")

✅ Pipeline core trích xuất 3D hoạt động ổn định!


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1779031516.150186     139 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779031516.175521     139 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779031516.176923     142 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779031516.177077     140 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779031516.177583     141 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779031516.187521     

In [4]:
import os
import shutil
import numpy as np

# THIẾT LẬP LƯỢT CHẠY TẠI ĐÂY (Nhập giá trị: 1, 2, 3, hoặc 4)
CURRENT_BATCH = 1  

KAGGLE_INPUT_DIR = "/kaggle/input/datasets/bowboochua9/vsl400-splited/data_splited/train"
OUTPUT_KEYPOINTS_DIR = "/kaggle/working/keypoints"
TEST_VIS_DIR = "/kaggle/working/test_vis"

if os.path.exists(OUTPUT_KEYPOINTS_DIR):
    shutil.rmtree(OUTPUT_KEYPOINTS_DIR)
    print("🧹 Đã dọn sạch dữ liệu cũ lưu trên đĩa.")

if os.path.exists(TEST_VIS_DIR):
    shutil.rmtree(TEST_VIS_DIR)

if os.path.exists('/kaggle/working'):
    for item in os.listdir('/kaggle/working'):
        item_path = os.path.join('/kaggle/working', item)
        if not item.startswith('.'):
            if os.path.isdir(item_path): shutil.rmtree(item_path)
            else: os.remove(item_path)

all_classes = sorted([f for f in os.listdir(KAGGLE_INPUT_DIR) if os.path.isdir(os.path.join(KAGGLE_INPUT_DIR, f))])
total_classes = len(all_classes)
chunks = np.array_split(all_classes, 4)
active_classes = list(chunks[CURRENT_BATCH - 1])

print(f"📊 Dataset gốc: {total_classes} classes.")
print(f"🚀 TIẾN TRÌNH ĐANG XỬ LÝ: BATCH SỐ {CURRENT_BATCH}/4")
print(f"🔥 Số lượng danh mục phụ trách: {len(active_classes)} classes (Từ '{active_classes[0]}' đến '{active_classes[-1]}').")

📊 Dataset gốc: 400 classes.
🚀 TIẾN TRÌNH ĐANG XỬ LÝ: BATCH SỐ 1/4
🔥 Số lượng danh mục phụ trách: 100 classes (Từ 'Anh' đến 'Dũng cảm').


In [ ]:
from glob import glob
from tqdm import tqdm
from pathlib import Path
import os

for class_name in active_classes:
    class_input_path = os.path.join(KAGGLE_INPUT_DIR, class_name)
    video_files = glob(os.path.join(class_input_path, "*.mp4"))
    print(f"\n📂 Tiến hành trích xuất: '{class_name}' ({len(video_files)} videos)")
    
    for video_path in tqdm(video_files, desc=f"Batch {CURRENT_BATCH} -> {class_name}"):
        video_id = Path(video_path).stem
        npy_out_path = os.path.join(OUTPUT_KEYPOINTS_DIR, class_name, f"{video_id}.npy")
        
        if os.path.exists(npy_out_path):
            continue
            
        extract_sign_language_features(video_path, npy_out_path, visualize=False)

print(f"\n🎉 HOÀN THÀNH XỬ LÝ TOÀN BỘ BATCH SỐ {CURRENT_BATCH}/4!")

In [ ]:
import os
from glob import glob
from pathlib import Path

if 'active_classes' in locals() and len(active_classes) > 0:
    test_class = active_classes[0]
    test_class_path = os.path.join(KAGGLE_INPUT_DIR, test_class)
    test_videos = sorted(glob(os.path.join(test_class_path, "*.mp4")))
    
    if test_videos:
        videos_to_test = test_videos[:10]
        print(f"🎬 Chế độ Test Diện Rộng: Trích xuất mồi cho 10 video của lớp '{test_class}':")
        os.makedirs("/kaggle/working/test_vis", exist_ok=True)
        
        for idx, video_test_path in enumerate(videos_to_test):
            video_test_id = Path(video_test_path).stem
            npy_test_out = os.path.join(OUTPUT_KEYPOINTS_DIR, test_class, f"{video_test_id}.npy")
            video_vis_out = os.path.join("/kaggle/working/test_vis", f"{video_test_id}_overlay.mp4")
            
            print(f"  📹 [{idx+1}/10] Đang xử lý: {Path(video_test_path).name} ...", end="")
            
            if os.path.exists(npy_test_out): os.remove(npy_test_out)
            if os.path.exists(video_vis_out): os.remove(video_vis_out)
                
            thành_công = extract_sign_language_features(
                video_test_path, 
                npy_test_out, 
                visualize=True, 
                vis_out_path=video_vis_out
            )
            if thành_công: print(" ✅ Xong!")
            else: print(" ❌ Thất bại!")
                
        print("\n🎉 Đã trích xuất xong 10 video mẫu thử nghiệm!")

🎬 Chế độ Test Diện Rộng: Trích xuất mồi cho 10 video của lớp 'Anh':
  📹 [1/10] Đang xử lý: 000861.mp4 ...

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


 ✅ Xong!
  📹 [2/10] Đang xử lý: 000862.mp4 ... ✅ Xong!
  📹 [3/10] Đang xử lý: 001702.mp4 ... ✅ Xong!
  📹 [4/10] Đang xử lý: 001703.mp4 ... ✅ Xong!
  📹 [5/10] Đang xử lý: 003414.mp4 ... ✅ Xong!
  📹 [6/10] Đang xử lý: 003415.mp4 ... ✅ Xong!
  📹 [7/10] Đang xử lý: 003416.mp4 ...

In [ ]:
import os
import cv2
import numpy as np
from glob import glob
from pathlib import Path
from IPython.display import Video, display, HTML

def tạo_video_normalized_motion_bulk(npy_path, out_video_path, fps=25):
    BODY_LANDMARKS_LOCAL = ["nose", "leftEyeInner", "leftEye", "leftEyeOuter", "rightEyeInner", "rightEye", "rightEyeOuter", "leftEar", "rightEar", "mouthLeft", "mouthRight", "leftShoulder", "rightShoulder", "leftElbow", "rightElbow", "leftWrist", "rightWrist", "leftPinky", "rightPinky", "leftIndex", "rightIndex", "leftThumb", "rightThumb", "leftHip", "rightHip", "leftKnee", "rightKnee", "leftAnkle", "rightAnkle", "leftHeel", "rightHeel", "leftFootIndex", "rightFootIndex", "neck"]
    HAND_LANDMARKS_LOCAL = ["wrist", "indexTip", "indexDIP", "indexPIP", "indexMCP", "middleTip", "middleDIP", "middlePIP", "middleMCP", "ringTip", "ringDIP", "ringPIP", "ringMCP", "littleTip", "littleDIP", "littlePIP", "littleMCP", "thumbTip", "thumbIP", "thumbMP", "thumbCMC"]
    LANDMARKS_LOCAL = BODY_LANDMARKS_LOCAL + [id + sfx for id in HAND_LANDMARKS_LOCAL for sfx in ["_0", "_1"]]
    LM_IDX = {name: i for i, name in enumerate(LANDMARKS_LOCAL)}

    data = np.load(npy_path)
    total_frames = len(data)
    S = 300
    
    os.makedirs(os.path.dirname(out_video_path), exist_ok=True)
    out_writer = cv2.VideoWriter(out_video_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (S * 3, S))

    BODY_CONNECTIONS = [
        ("nose", "neck"), ("neck", "rightShoulder"), ("neck", "leftShoulder"),
        ("rightShoulder", "rightElbow"), ("rightElbow", "rightWrist"),
        ("leftShoulder", "leftElbow"), ("leftElbow", "leftWrist"),
        ("rightShoulder", "leftShoulder"),
        ("leftShoulder", "leftHip"), ("rightShoulder", "rightHip"),
        ("leftHip", "rightHip"),
        ("leftHip", "leftKnee"), ("leftKnee", "leftAnkle"),
        ("rightHip", "rightKnee"), ("rightKnee", "rightAnkle")
    ]
    FINGER_CHAINS = [["wrist", "thumbCMC", "thumbMP", "thumbIP", "thumbTip"], ["wrist", "indexMCP", "indexPIP", "indexDIP", "indexTip"], ["wrist", "middleMCP", "middlePIP", "middleDIP", "middleTip"], ["wrist", "ringMCP", "ringPIP", "ringDIP", "ringTip"], ["wrist", "littleMCP", "littlePIP", "littleDIP", "littleTip"]]

    for t in range(total_frames):
        kp_frame = data[t] + 0.5
        img_body = np.zeros((S, S, 3), dtype=np.uint8)
        img_left_hand = np.zeros((S, S, 3), dtype=np.uint8)
        img_right_hand = np.zeros((S, S, 3), dtype=np.uint8)

        def to_pixels(name):
            if name not in LM_IDX: return None
            x, y, z = kp_frame[LM_IDX[name]]
            if abs(x - 0.5) < 1e-4 and abs(y - 0.5) < 1e-4: return None
            return (int(x * (S - 30)) + 15, int(y * (S - 30)) + 15)

        for a, b in BODY_CONNECTIONS:
            pa, pb = to_pixels(a), to_pixels(b)
            if pa and pb:
                cv2.line(img_body, pa, pb, (0, 255, 0), 2)
                cv2.circle(img_body, pa, 4, (0, 0, 255), -1)

        for suffix, canvas, color in [("_0", img_left_hand, (255, 165, 0)), ("_1", img_right_hand, (0, 165, 255))]:
            for chain in FINGER_CHAINS:
                for i in range(len(chain) - 1):
                    pa = to_pixels(chain[i] + suffix)
                    pb = to_pixels(chain[i+1] + suffix)
                    if pa and pb:
                        cv2.line(canvas, pa, pb, color, 2)
                        cv2.circle(canvas, pa, 3, (255, 255, 255), -1)

        cv2.putText(img_body, "1. Full Body Norm (3D)", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        cv2.putText(img_left_hand, "2. Left Hand Norm", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 165, 0), 2)
        cv2.putText(img_right_hand, "3. Right Hand Norm", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 165, 255), 2)

        combined_frame = np.hstack((img_body, img_left_hand, img_right_hand))
        out_writer.write(combined_frame)
        
    out_writer.release()

vis_folder = "/kaggle/working/test_vis"
if 'test_class' in locals() and os.path.exists(OUTPUT_KEYPOINTS_DIR):
    npy_files = sorted(glob(os.path.join(OUTPUT_KEYPOINTS_DIR, test_class, "*.npy")))[:10]
    for idx, npy_path in enumerate(npy_files):
        v_id = Path(npy_path).stem
        overlay_video_path = os.path.join(vis_folder, f"{v_id}_overlay.mp4")
        norm_motion_path = os.path.join(vis_folder, f"{v_id}_normalized_motion.mp4")
        
        display(HTML(f"<div style='background-color: #222; padding: 10px; margin-top: 40px; border-left: 5px solid #00E676;'>"
                     f"<h3 style='color: white; margin: 0;'>🎬 VIDEO KIỂM THỬ SỐ {idx+1}/10 — ID: {v_id}.mp4</h3></div>"))
        
        print(f"📺 Tầng 1: Xương MediaPipe thô đè video gốc")
        if os.path.exists(overlay_video_path): display(Video(overlay_video_path, embed=True, width=700))
            
        tạo_video_normalized_motion_bulk(npy_path, norm_motion_path, fps=25)
        print(f"📊 Tầng 2: Hoạt cảnh 2D chiếu từ Ma trận dữ liệu 3D")
        if os.path.exists(norm_motion_path): display(Video(norm_motion_path, embed=True, width=700))